# 실제 의료 데이터(심장 질환)를 활용한 PyTorch 이진 분류 모델

인공지능개론 프로젝트: 학습 전/후 인식률과 손실 변화를 비교합니다.


## 1. 프로젝트 목표

- 13개 환자 특징으로 심장 질환 여부를 0/1로 예측합니다.
- 학습 전 무작위 모델과 학습 후 모델의 정확도, 재현율을 비교합니다.
- 12장 Dropout, 13장 이진 분류, 14장 정규화 내용을 적용합니다.


In [ ]:
from heart_disease_project import *
from IPython.display import Image, display

set_seed(42)
ensure_dirs()


## 2. 데이터 불러오기와 결측치 확인

계획서의 데이터셋 설명에 맞춰 `data/heart.csv` 파일을 실제로 남겨두고, Pandas의 `pd.read_csv()`로 CSV를 불러옵니다. 이 CSV는 Kaggle Heart Disease Dataset에서 쓰이는 13개 특징 + target 형식과 맞춘 파일입니다.


In [ ]:
# Pandas를 이용한 CSV 파일 불러오기: 평가 과정에서 확인할 핵심 코드입니다.
raw_df = pd.read_csv(DATA_DIR / "heart.csv")
clean_df = clean_dataset(raw_df)

print(raw_df.shape)
print(raw_df.head())
print(raw_df.isna().sum())
print(clean_df[TARGET_COLUMN].value_counts().sort_index())
print("CSV file:", DATA_DIR / "heart.csv")
print("Raw missing report:", RAW_MISSING_REPORT_CSV)
print("Missing report:", MISSING_REPORT_CSV)
print("Clean CSV:", CLEAN_CSV)


## 3. EDA 시각화

클래스 분포와 주요 변수의 상관관계를 확인합니다.


In [ ]:
plot_class_distribution(clean_df)
plot_correlation_heatmap(clean_df)

display(Image(filename=str(OUTPUT_DIR / "class_distribution.png")))
display(Image(filename=str(OUTPUT_DIR / "correlation_heatmap.png")))


## 4. 학습/검증/테스트 분리와 정규화

정답 라벨은 제외하고 `df.iloc[:, :-1]`에 해당하는 특징 데이터만 표준화합니다. 평균과 표준편차는 훈련 데이터에서만 계산해 검증/테스트 데이터에 적용합니다. 분리된 CSV와 정규화된 CSV도 `data/` 폴더에 저장합니다.


In [ ]:
train_df, valid_df, test_df = split_dataframe(clean_df)
train_data_pd, valid_data_pd, test_data_pd, mean, std = standardize_features(train_df, valid_df, test_df)

x_train, y_train = to_tensor(*train_data_pd)
x_valid, y_valid = to_tensor(*valid_data_pd)
x_test, y_test = to_tensor(*test_data_pd)

print("Train:", x_train.shape, y_train.shape)
print("Valid:", x_valid.shape, y_valid.shape)
print("Test:", x_test.shape, y_test.shape)
print("Saved split CSV:", TRAIN_CSV, VALID_CSV, TEST_CSV)
print("Saved normalized CSV:", NORMALIZED_TRAIN_CSV, NORMALIZED_VALID_CSV, NORMALIZED_TEST_CSV)
print("Saved normalization info:", NORMALIZATION_INFO_CSV)


## 5. PyTorch 모델 설계

입력층 13개에서 시작해 은닉층을 거쳐 마지막에 Sigmoid로 0~1 확률을 출력합니다. 손실 함수는 이진 분류에 맞는 `nn.BCELoss`, 최적화 함수는 Adam을 사용합니다.


In [ ]:
model = HeartDiseaseClassifier(input_size=len(FEATURE_COLUMNS), dropout_p=0.25)
criterion = nn.BCELoss()
print(model)


## 6. 학습 전 성능 측정

가중치가 무작위로 초기화된 상태에서 테스트 성능을 먼저 측정합니다.


In [ ]:
before_training = evaluate(model, x_test, y_test, criterion)
before_training


## 7. 모델 학습

검증 손실이 오래 개선되지 않으면 조기 종료합니다.


In [ ]:
history = train_model(model, (x_train, y_train), (x_valid, y_valid), n_epochs=300, lr=0.001, patience=45)
plot_training_history(history)

display(Image(filename=str(OUTPUT_DIR / "loss_curve.png")))
display(Image(filename=str(OUTPUT_DIR / "valid_metrics_curve.png")))

print("epochs used:", len(history["epoch"]))
print("best valid loss:", min(history["valid_loss"]))


## 8. 학습 후 성능 평가와 비교

의료 데이터에서는 실제 환자를 정상으로 오판하지 않는 것이 중요하므로 정확도와 함께 재현율을 확인합니다.


In [ ]:
after_training = evaluate(model, x_test, y_test, criterion)
plot_before_after(before_training, after_training)
display(Image(filename=str(OUTPUT_DIR / "before_after_metrics.png")))

print("Before:", before_training)
print("After:", after_training)


## 9. 결론

위 셀에서 EDA 그래프, 학습 곡선, 학습 전/후 비교 그래프를 노트북 안에서 바로 확인할 수 있습니다. 같은 결과는 `outputs/metrics.json`과 발표용 PPT에도 정리되어 있습니다. 데이터 수가 303개로 적기 때문에 과적합 위험이 있으며, 향후에는 더 많은 의료 데이터 확보, 교차 검증, threshold 조정으로 재현율을 개선할 수 있습니다.
